<a href="https://colab.research.google.com/github/MayaVolkov/astronaut-missions-analysis/blob/main/Astronaut_Mission_Analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Astronaut Mission Analysis (from 1961 to 2026)

## Question
How has the number of countries sending people to space changed since 1961?

Sub-questions:
1. How do countries without their own human launch rockets get their astronauts there?
2. How has participation in human spaceflight by countries outside the three launch nations changed since 1961?

## Data:
My data is coming from two sources: Mission data from 1961 to early 2020 is available through a public database published by Mariya Stavnichuk and Tatsuya Corlett in 2020. I got this dataset from TidyTuesday, a public github repo that publishes clean datasets for people to practice on. This dataset only goes until 2020, and I to answer my question I need the most curennt data as well. For this reason I am using a second dataset, for the Missions flown between 2021 and 2026 I will be using the public Wikipedia page to get information on flights happening after early 2020 into 2021.

## Scope:
I have chosen to include all orbital flights between 1961 and 2026. 1961 is when the first human space mission occured, and going until 2026 will allow me to get the most up to date data. I chose not to include suborbital flights as based on my research those are typically commerical spaceflights that arent tied to a nation or space research institute and can be attributed to "space tourism".

## Why does this matter?
This matters because space is one of the last frontiers of truly wilderness and unknown, and making space technology more accessible to other nations allows for there to be more minds and global energy directed into research into space.

In [124]:
import pandas as pd

import requests

from io import StringIO

import re

Below I am storing the link to the data for the missions between 1961 and early 2020. Im storing it in a variable so that its easy for me to grab whenever I need. Instead of copying in the whole link over again, I can just type the variable.

In [125]:
data_url_until_2020 = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-07-14/astronauts.csv"
print(type(data_url_until_2020))

<class 'str'>


Using Pandas to load in the csv file and turn it into a table that colab can read. Then I print the five *rows* of the table as a sanity check.

In [126]:
data_table_2020 = pd.read_csv(data_url_until_2020)
#peeking
data_table_2020.head()

,id,number,nationwide_number,name,original_name,sex,year_of_birth,nationality,military_civilian,selection,...,year_of_mission,mission_title,ascend_shuttle,in_orbit,descend_shuttle,hours_mission,total_hrs_sum,field21,eva_hrs_mission,total_eva_hrs
0,1,1,1,"Gagarin, Yuri",ГАГАРИН Юрий Алексеевич,male,1934,U.S.S.R/Russia,military,TsPK-1,...,1961,Vostok 1,Vostok 1,Vostok 2,Vostok 3,1.77,1.77,0,0.0,0.0
1,2,2,2,"Titov, Gherman",ТИТОВ Герман Степанович,male,1935,U.S.S.R/Russia,military,TsPK-1,...,1961,Vostok 2,Vostok 2,Vostok 2,Vostok 2,25.00,25.30,0,0.0,0.0
2,3,3,1,"Glenn, John H., Jr.","Glenn, John H., Jr.",male,1921,U.S.,military,NASA Astronaut Group 1,...,1962,MA-6,MA-6,MA-6,MA-6,5.00,218.00,0,0.0,0.0
3,4,3,1,"Glenn, John H., Jr.","Glenn, John H., Jr.",male,1921,U.S.,military,NASA Astronaut Group 2,...,1998,STS-95,STS-95,STS-95,STS-95,213.00,218.00,0,0.0,0.0
4,5,4,2,"Carpenter, M. Scott","Carpenter, M. Scott",male,1925,U.S.,military,NASA- 1,...,1962,Mercury-Atlas 7,Mercury-Atlas 7,Mercury-Atlas 7,Mercury-Atlas 7,5.00,5.00,0,0.0,0.0


Im going to be focusing on analyzing the following four columns of data for the following reasons:
- nationality: in order to find the countries who are sending people to space
- name: im not just looking at countries I am looking at the individual people who were associated with those countries so I need those.
- year_of_mission: im checking the changes of countries over time
- ascend_shuttle: tells me which spacecraft launched that person so I know which countries rocket carried them.  


research decisions:
- Only count astronauts who reached orbit (aborted launches and suborbital flights excluded), because the question is about arrivals, not attempts.
- Treat the USSR and Russia as one country, since they never overlapped and share one continuous space program.

Open question: the dataset labels some Soviet astronauts U.S.S.R/Ukraine and others U.S.S.R/Russia. Check those rows in Lesson 7 before deciding how to count them.

In [127]:
analysis_columns = ["name", "year_of_mission", "nationality", "ascend_shuttle"]

data_table_2020[analysis_columns].head(5)


,name,year_of_mission,nationality,ascend_shuttle
0,"Gagarin, Yuri",1961,U.S.S.R/Russia,Vostok 1
1,"Titov, Gherman",1961,U.S.S.R/Russia,Vostok 2
2,"Glenn, John H., Jr.",1962,U.S.,MA-6
3,"Glenn, John H., Jr.",1998,U.S.,STS-95
4,"Carpenter, M. Scott",1962,U.S.,Mercury-Atlas 7


Creating a dictionary that stores the key User-Agent and my contact info so I can request information from Wikipedia because Wikipedia asks code to identify itself before it sends pages.

In [128]:
headers = {"User-Agent" : "Astronaut Mission Analysis Student Project (https://github.com/MayaVolkov)"}

Saving the wiki URL and then requesting the information inside the page, attaching my username to the request so wikipedia knows where its coming from. Status_code reports if we got the file -> 200 means success.

In [129]:
wiki_url = "https://en.wikipedia.org/wiki/List_of_human_spaceflights,_2021%E2%80%93present"
response = requests.get(wiki_url, headers=headers)
response.status_code

200

creating a variable called women to hold all of the data in the dataframe pertaining to female flyers.

In [130]:
women= data_table_2020[data_table_2020["sex"] == "female"]


In [131]:
women.shape

(143, 24)

In [132]:
women["name"].nunique()

64

creating a variable after_2000 holding all the data pertaining to flights that happened after the year 2000

In [133]:
after_2000 = data_table_2020[data_table_2020["year_of_mission"] > 2000]

Astronauts from 23 different nationality labels flew on missions after 2000

In [134]:
after_2000["nationality"].nunique()

23

####Research decision: how to count Soviet-era nationality labels.
 The U.S.S.R/Ukraine row (Kadenyuk, 1997) is counted as Ukraine. His flight came after Ukraine’s independence in 1991, he flew as a citizen of Ukraine, and he flew on a U.S. spacecraft, so it isn’t a continuation of the Soviet program.

In [135]:
ukraine = data_table_2020[data_table_2020["nationality"] == "U.S.S.R/Ukraine"]

In [136]:
ukraine_columns = ["name" , "year_of_mission" , "ascend_shuttle"]
ukraine[ukraine_columns].head(10)

,name,year_of_mission,ascend_shuttle
913,"Kadenyuk, Leonid",1997,STS87


Seeing what cells are empty in the whole table:

In [137]:
data_table_2020.isnull().sum()

,0
id,0
number,0
nationwide_number,0
name,0
original_name,5
sex,0
year_of_birth,0
nationality,0
military_civilian,0
selection,5


seeing what cells are missing just from the columns im using for my research

In [138]:
research_cols = ["name", "nationality", "year_of_mission", "ascend_shuttle"]
data_table_2020[research_cols].isnull().sum()

,0
name,0
nationality,0
year_of_mission,0
ascend_shuttle,1


The only column with missing data is ascend_shuttle, which has one cell in its column missing. Because there is only one missing cell I will keep that row and manually fill it in.

In [139]:
data_table_2020[data_table_2020["ascend_shuttle"].isnull()]

,id,number,nationwide_number,name,original_name,sex,year_of_birth,nationality,military_civilian,selection,...,year_of_mission,mission_title,ascend_shuttle,in_orbit,descend_shuttle,hours_mission,total_hrs_sum,field21,eva_hrs_mission,total_eva_hrs
197,198,94,1,Bertalan Farkas,Farkas Bertalan,male,1949,Hungry,civilian,1978 Intercosmos Group,...,1980,Soyuz 36/35,NaN,Soyuz 36/35,NaN,188.75,188.75,0,0.0,0.0


The missing text cell is for Bertalan Farkas, 1980, he launched on Soyuz 36 and landed back on Earth on Soyuz 35. Source https://www.britannica.com/biography/Bertalan-Farkas

In [140]:
no_comma_count = (~(data_table_2020["name"].str.contains(","))).sum()
no_comma_count

np.int64(33)

In [141]:
withoutComma = data_table_2020[~data_table_2020["name"].str.contains(",")]
withoutComma

,id,number,nationwide_number,name,original_name,sex,year_of_birth,nationality,military_civilian,selection,...,year_of_mission,mission_title,ascend_shuttle,in_orbit,descend_shuttle,hours_mission,total_hrs_sum,field21,eva_hrs_mission,total_eva_hrs
67,68,101,1,Jugderdemidiin Gurragchaa,Жүгдэрдэмидийн Гүррагчаа,male,1947,Mongolia,civilian,1978 Intercosmos Group,...,1981,Soyuz 39,Soyuz 39,Soyuz 39,Soyuz 39,188.70,188.70,0,0.00,0.00
72,73,103,1,Dumitru Prunariu,Dumitru-Dorin Prunariu,male,1952,Romania,civilian,1978 Intercosmos Group,...,1981,Soyuz 40,Soyuz 40,Soyuz 40,Soyuz 40,188.70,188.70,0,0.00,0.00
185,186,87,1,Vladimir Remek,NaN,male,1948,Czechoslovakia,military,Air Gorce Group 6 - USSR,...,1978,Soyuz 28,Soyuz 28,Soyuz 28,Soyuz 28,190.28,190.28,0,0.00,0.00
188,189,89,1,Miroslaw Hermaszewske,Miroslaw Hermaszewske,male,1941,Poland,military,1976 Intercosmos Group,...,1978,Soyuz 30,Soyuz 30,Soyuz 30,Soyuz 30,190.03,190.03,0,0.00,0.00
193,194,92,1,Georgi Ivanov,Георги Иванов Иванов,male,1940,Bulgaria,military,1978 Intercosmos Group,...,1979,Soyuz 33,Soyuz 33,Soyuz 33,Soyuz 33,47.00,47.00,0,0.00,0.00
197,198,94,1,Bertalan Farkas,Farkas Bertalan,male,1949,Hungry,civilian,1978 Intercosmos Group,...,1980,Soyuz 36/35,NaN,Soyuz 36/35,NaN,188.75,188.75,0,0.00,0.00
200,201,96,1,Pham Tuan,Phạm Tuân,male,1947,Vietnam,military,1979 Intercosmos Group,...,1980,Salyut 6,Soyuz 37,Salyut 6,Soyuz 36,188.70,188.70,0,0.00,0.00
201,202,97,1,Arnaldo Tamayo Mendez,Arnaldo Tamayo Méndez,male,1942,Cuba,military,1978 Intercosmos Group,...,1980,Soyuz 38,Soyuz 38,Soyuz 38,Soyuz 38,1887.71,188.71,0,0.00,0.00
285,286,138,1,Rakesh Sharma,राकेश शर्मा,male,1949,India,military,1982 Soviet Intercosmos Space Program,...,1984,Salyut 7,Soyuz T-10,Salyut 7,Soyuz T-11,4365.80,4365.80,0,0.00,0.00
570,571,252,1,Helen Sharman,Helen Sharman,female,1963,U.K.,civilian,1989 Juno,...,1991,Soyuz TM-12/11,Soyuz TM-12,Soyuz TM-12,Soyuz TM-11,189.22,189.22,0,0.00,0.00


In [142]:
data_table_2020["name_clean"] = (
    data_table_2020["name"]
    .str.replace("Kregel Kevin R.", "Kevin R. Kregel")
    .str.replace("Hermaszewske", "Hermaszewski")
    .str.replace(" (Aydyn) ", " ")
)

###Decision:
keep names as the dataset has them. I only fixed clear errors (name order, typos, and an alternate spelling in parentheses). I did not add or remove middle names, middle initials, or patronymics, because changing them would mean creating versions of names that the source doesn’t use. Names will have different numbers of parts (for example, Yuri Vladimirovich Usachyov vs. Helen Sharman), but that’s fine: when I link datasets later, the match key will only use the first and last parts, so whatever’s in the middle won’t affect matching.

In [143]:
changed = data_table_2020["name"] != data_table_2020["name_clean"]
data_table_2020[changed][["name", "name_clean"]]

,name,name_clean
188,Miroslaw Hermaszewske,Miroslaw Hermaszewski
807,Kregel Kevin R.,Kevin R. Kregel
808,Kregel Kevin R.,Kevin R. Kregel
809,Kregel Kevin R.,Kevin R. Kregel
810,Kregel Kevin R.,Kevin R. Kregel
1253,Aidyn (Aydyn) Akanovich Aimbetov,Aidyn Akanovich Aimbetov


Flipped ‘Last, First’ names to ‘First Last’. Names with a suffix (like Jr.) keep it at the end. Names without a comma were left as they were.”

In [144]:
has_comma = data_table_2020["name_clean"].str.contains(",")
pieces = data_table_2020["name_clean"].str.split(", ")

last = pieces.str[0]
first = pieces.str[1]
suffix = pieces.str[2].fillna("")

flipped = (first + " " + last + " " + suffix).str.strip()
data_table_2020["name_clean"] = flipped.where(has_comma, data_table_2020["name_clean"])

In [145]:
data_table_2020[["name", "name_clean"]].head(10)

,name,name_clean
0,"Gagarin, Yuri",Yuri Gagarin
1,"Titov, Gherman",Gherman Titov
2,"Glenn, John H., Jr.",John H. Glenn Jr.
3,"Glenn, John H., Jr.",John H. Glenn Jr.
4,"Carpenter, M. Scott",M. Scott Carpenter
5,"Nikolayev, Andriyan",Andriyan Nikolayev
6,"Nikolayev, Andriyan",Andriyan Nikolayev
7,"Popovich, Pavel",Pavel Popovich
8,"Popovich, Pavel",Pavel Popovich
9,"Schirra, Walter M., Jr.",Walter M. Schirra Jr.


In [146]:
data_table_2020["name_clean"].str.contains(",").sum()

np.int64(0)

Collecting 2021–2026 missions. Source: Wikipedia, “List of human spaceflights, 2021–present” (link). Each mission is a table row (`<tr>`), with astronaut names inside links (`<a>`). Suborbital flights are marked with a green background (style="background:#BBFFBB") and will be excluded. Names include a flight count in parentheses, e.g. “Oleg Novitsky (3)”, which will need to be removed.

In [147]:
wrapped_html = StringIO(response.text)
tables = pd.read_html(wrapped_html)
print(len(tables))

7


In [148]:
for i, table in enumerate(tables):
    print(i, table.shape)

0 (76, 7)
1 (22, 2)
2 (8, 2)
3 (4, 2)
4 (1, 2)
5 (1, 2)
6 (1, 2)


In [149]:
missions = tables[0]
missions.columns

Index(['#', 'Crew', 'Launch spacecraft', 'Habitation', 'Habitation.1',
       'Return spacecraft', 'Brief mission summary'],
      dtype='object')

In [150]:
missions.head()

,#,Crew,Launch spacecraft,Habitation,Habitation.1,Return spacecraft,Brief mission summary
0,329,Oleg Novitsky (3),9 April 2021 Soyuz MS-18,ISS (crew 64/65),ISS (crew 64/65),17 October 2021 Soyuz MS-18,ISS crew rotation.
1,329,Pyotr Dubrov (1) Mark T. Vande Hei (2),9 April 2021 Soyuz MS-18,ISS (crew 64/65/66),ISS (crew 64/65/66),30 March 2022 Soyuz MS-19,ISS crew rotation.
2,330,Shane Kimbrough (3) K. Megan McArthur (2) Ak...,"23 April 2021 SpaceX Crew-2, Endeavour",ISS (crew 65/66),ISS (crew 65/66),"9 November 2021 SpaceX Crew-2, Endeavour",ISS crew rotation.
3,—,Frederick W. Sturckow Dave Mackay,22 May 2021 Unity 21,22 May 2021 Unity 21,22 May 2021 Unity 21,22 May 2021 Unity 21,First human spaceflight from the State of New ...
4,331,Nie Haisheng (3) Liu Boming (2) Tang Hongbo (1),17 June 2021 Shenzhou 12,TSS,TSS,17 September 2021 Shenzhou 12,First crew to Tiangong Space Station.


used pd.read_html to process the html file into tables (dataframes), from looking at the different tables it outputted I knew the mission data was most likely going to be in table[0] because it had the most columns (aka most types of information) and the most rows (most data), pd.read_html throws away anything not considered text, like the cells background color or the flags next to crew members names. I confirmed my hypothesis by running mission.head()
3 suprises:
1. two habitation headers - The Habitation header on the page sits over two columns, so pandas needs two column names. It can’t have two columns both called “Habitation,” so it adds .1 to the second one to keep the names unique.
2. split rows sharing a # - Different people go up for different reasons, and on different missions. Wikipedia gives each return group its own row and copies the launch info (and the #) into both
3. the "-" on some flights - these, along with having same day launch and return dates, are our clue that these rows are suborbital fights, good to have since pd.read_html threw away the other signifier (green background)

In [151]:
flag_pattern = r'<[^>]*title="([^"]+)"[^>]*><img[^>]*>'
br_pattern = r'<br[^>]*>'

html_clean = re.sub(flag_pattern, r'[\1] ', response.text)
html_clean = re.sub(br_pattern, ' | ', html_clean)

# check: every country the flag pattern caught, duplicates removed
for country in sorted(set(re.findall(flag_pattern, response.text))):
    print(country)

Antigua and Barbuda
Australia
Austria
Belarus
Brazil
Canada
China
Denmark
Egypt
France
Germany
Hungary
India
Iran
Israel
Italy
Japan
Kazakhstan
Malta
Netherlands
New Zealand
Nigeria
Norway
Pakistan
Panama
Poland
Portugal
Puerto Rico
Russia
Saint Kitts and Nevis
Saudi Arabia
Singapore
South Africa
Spain
Sweden
The Bahamas
Turkey
Ukraine
United Arab Emirates
United Kingdom
United States


In [152]:
tables_clean = pd.read_html(StringIO(html_clean))
missions = tables_clean[0]
missions.head()

,#,Crew,Launch | spacecraft,Habitation,Habitation.1,Return | spacecraft,Brief mission summary
0,329,[Russia] Oleg Novitsky (3),9 April 2021 | [Russia] Soyuz MS-18,ISS | (crew 64/65),ISS | (crew 64/65),17 October 2021 | [Russia] Soyuz MS-18,ISS crew rotation.
1,329,[Russia] Pyotr Dubrov (1) | [United States] Ma...,9 April 2021 | [Russia] Soyuz MS-18,ISS | (crew 64/65/66),ISS | (crew 64/65/66),30 March 2022 | [Russia] Soyuz MS-19,ISS crew rotation.
2,330,[United States] Shane Kimbrough (3) | [United ...,"23 April 2021 | [United States] SpaceX Crew-2,...",ISS | (crew 65/66),ISS | (crew 65/66),9 November 2021 | [United States] SpaceX Crew-...,ISS crew rotation.
3,—,[United States] Frederick W. Sturckow [United ...,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21,22 May 2021 [United States] Unity 21,First human spaceflight from the State of New ...
4,331,[China] Nie Haisheng (3) | [China] Liu Boming ...,17 June 2021 | [China] Shenzhou 12,TSS,TSS,17 September 2021 | [China] Shenzhou 12,First crew to Tiangong Space Station.


In [153]:
missions["crew_clean"] = missions["Crew"].str.replace(r'\s*\(\d+\)', '', regex=True)
missions[["Crew", "crew_clean"]].head()

,Crew,crew_clean
0,[Russia] Oleg Novitsky (3),[Russia] Oleg Novitsky
1,[Russia] Pyotr Dubrov (1) | [United States] Ma...,[Russia] Pyotr Dubrov | [United States] Mark T...
2,[United States] Shane Kimbrough (3) | [United ...,[United States] Shane Kimbrough | [United Stat...
3,[United States] Frederick W. Sturckow [United ...,[United States] Frederick W. Sturckow [United ...
4,[China] Nie Haisheng (3) | [China] Liu Boming ...,[China] Nie Haisheng | [China] Liu Boming | [C...
